# loss mask 的覆盖范围写错了

Loss Mask 覆盖范围写错的核心原因，是混淆了「<font color='red'>标签索引</font>」与「<font color='red'>预测位索引</font>」的错位关系。 
- 自回归模型中，位置 t 的 Logits 预测的是位置 t+1 的 Token，因此标签区间 [s, e] 对应的预测位区间应是 [s-1, e-1]，必须左移一位。

## 具体举例说明
假设序列为 a b c d e f g（长度 L=7），其中：
- Prompt：a b（长度 Lp=2）
- Response：c d e f g（标签索引 [2, 6]）

模型输出 Logits 后，有效预测位为 [:-1]，即前 6 个位置，分别预测 b c d e f g。

### ✅ 正确写法

In [ ]:
# Response 标签区间 [2, 6] → 对应预测位 [1, 5]
loss_mask[..., Lp-1 : -1] = 1   # 即 loss_mask[..., 1:6] = 1

此时 mask 覆盖的位置是：

| 位置 | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|------|---|---|---|---|---|---|---|
| Token | a | b | c | d | e | f | g |
| 预测目标 | b | c | d | e | f | g | — |
| Loss Mask | 0 | 1 | 1 | 1 | 1 | 1 | 0 |

模型学习的是：用 b 预测 c、用 c 预测 d……用 f 预测 g，全部是 Response 区域的监督信号。

### ❌ 错误写法

In [ ]:
# 错误：直接用标签区间，未左移
loss_mask[..., Lp : -1] = 1     # 即 loss_mask[..., 2:6] = 1

此时 mask 覆盖的位置是：

| 位置 | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|------|---|---|---|---|---|---|---|
| Token | a | b | c | d | e | f | g |
| 预测目标 | b | c | d | e | f | g | — |
| Loss Mask | 0 | 0 | 1 | 1 | 1 | 1 | 0 |

问题：位置 1 的预测目标 c 是 Response 的第一个 Token，但被 mask 掉了；同时位置 6 没有下一个 Token，本就不该计入。结果是 Response 的首尾两个 Token 都没被监督到，模型学不到完整的回答生成能力。

## 多轮对话中的同类错误

多轮 SFT 中，每段 Assistant 回复的标签区间 [s_k, e_k] 同样需要映射到预测位 [s_k-1, e_k]。常见错误是写成 [s_k, e_k]，导致每段回复的第一个 Token 漏掉监督。

In [ ]:
# ✅ 正确
loss_mask[..., s_k-1 : e_k] = 1

# ❌ 错误（首 token 漏监督）
loss_mask[..., s_k : e_k] = 1

## 排查建议
训练时打印一下 loss_mask 的可视化，确认 mask 的起始位置是否比 Response 起始 Token 提前一位，就能快速定位这类 off-by-one 问题。